# SQL Agent (SQLite Demo) — Step-by-Step Walkthrough

**Blauw-Zwart Analytics Pipeline**

This notebook walks through the SQL agent pipeline using a **self-contained
SQLite database** — no Docker, Postgres, or Kafka required. A full season of
synthetic fan events is generated from `match_day.example.json` and loaded into
two lightweight mart tables.

An English-language question enters, a validated SQL query is generated and
executed, and a Markdown answer comes back — all without the user ever writing
SQL.

---

### Pipeline at a glance

```
User question
      │
      ▼
┌─────────────────────────────────────────────────────┐
│            SQLite Demo Agent  (ReAct loop)          │
│                                                     │
│  list_tables → describe_table → search_columns      │
│  → sample_table → execute_select ✓                  │
│                                                     │
│  ┌──────────────────────────────────────────────┐   │
│  │         SQL Guardrails (3 layers)            │   │
│  │  1. strip_fences()                           │   │
│  │  2. rewrite_schema_qualifiers()              │   │
│  │  3. validate_sql()  (sqlglot AST + regex)    │   │
│  └──────────────────────────────────────────────┘   │
└──────────────────────┬──────────────────────────────┘
                       │
                       ▼
               SQLite (in-memory)
               LIMIT 100 safety net
                       │
                       ▼
               Markdown answer
```

### Steps covered in this notebook

| Step | What you'll see |
|------|----------------|
| **0. Setup** | Generate full-season data, create SQLite mart tables |
| **1. Schema discovery** | `list_tables`, `describe_table`, `search_columns`, `sample_table` |
| **2. SQL guardrails** | Fence stripping, schema rewriting, AST + regex validation |
| **3. Safe execution** | `execute_select` through the full guardrail pipeline |
| **4. Full walkthrough** | End-to-end agent-like tool-call sequence |
| **5. ReAct agent** | *(optional, needs API key)* Live agent answering a question |

**Key libraries:**
| Library | Role |
|---------|------|
| `langchain` / `langgraph` | ReAct agent loop and tool-calling framework |
| `sqlglot` | SQL parsing, AST validation, and schema rewriting |
| `sqlite3` | Lightweight local database (stdlib — no install needed) |
| `fan_events.generation.v2_calendar` | Synthetic match-day event generation |

---
## 0 · Setup — Generate Data & Create SQLite Database

Generate a full season of synthetic fan events from the 30-match calendar in
`match_day.example.json`, then load them into two minimal mart tables in an
in-memory SQLite database.

**No prerequisites** — runs entirely on the standard library + repo code.

In [ ]:
import json
import random
import sqlite3
import sys
from pathlib import Path


def _find_repo_root(start: Path) -> Path:
    """Walk upwards until we find the repository root markers."""
    markers = ("pyproject.toml", "dbt_project.yml", "docker-compose.yml")
    for candidate in (start, *start.parents):
        if all((candidate / marker).exists() for marker in markers):
            return candidate
    raise FileNotFoundError(
        "Could not locate the repository root from the current working directory."
    )


# ── Resolve paths and make project importable ──────────────────────────
REPO_ROOT = _find_repo_root(Path.cwd().resolve())
SRC_PATH = REPO_ROOT / "src"
if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

print(f"✅ Repo root: {REPO_ROOT}")
print(f"   src/ added to sys.path")

# ── Load the match calendar ──────────────────────────────────────────
from fan_events.generation.v2_calendar import (
    filter_matches_by_date_range,
    generate_v2_records,
    load_calendar_json,
    validate_and_parse_matches,
)

calendar_path = REPO_ROOT / "match_day.example.json"
doc = load_calendar_json(calendar_path)
parsed_matches = validate_and_parse_matches(doc)
print(f"✅ Loaded {len(parsed_matches)} matches from {calendar_path.name}")

# ── Generate synthetic events with a fixed seed ──────────────────────
rng = random.Random(42)
contexts = filter_matches_by_date_range(parsed_matches, from_date=None, to_date=None)
records = generate_v2_records(contexts, rng, fan_pool_max=30_000)
print(f"✅ Generated {len(records):,} fan events across {len(contexts)} matches")

# ── Create in-memory SQLite database ─────────────────────────────────
db = sqlite3.connect(":memory:")
db.row_factory = sqlite3.Row
cur = db.cursor()

# Table 1: match_events — one row per generated event
cur.execute("""
    CREATE TABLE match_events (
        event_id   INTEGER PRIMARY KEY AUTOINCREMENT,
        fan_id     TEXT    NOT NULL,
        event_type TEXT    NOT NULL,
        match_id   TEXT    NOT NULL,
        timestamp  TEXT    NOT NULL,
        amount     REAL,
        item       TEXT,
        location   TEXT
    )
""")

for rec in records:
    cur.execute(
        "INSERT INTO match_events "
        "(fan_id, event_type, match_id, timestamp, amount, item, location) "
        "VALUES (?, ?, ?, ?, ?, ?, ?)",
        (
            rec["fan_id"],
            rec["event"],
            rec["match_id"],
            rec["timestamp"],
            rec.get("amount"),
            rec.get("item"),
            rec.get("location", ""),
        ),
    )

# Table 2: fan_loyalty — aggregated mart (one row per fan)
cur.execute("""
    CREATE TABLE fan_loyalty AS
    SELECT
        fan_id,
        COUNT(DISTINCT CASE WHEN event_type = 'ticket_scan' THEN match_id END)
            AS matches_attended,
        COALESCE(SUM(CASE WHEN event_type = 'merch_purchase' THEN amount END), 0.0)
            AS total_spend,
        COALESCE(SUM(CASE WHEN event_type = 'merch_purchase' THEN amount END), 0.0)
            AS merch_spend,
        COUNT(CASE WHEN event_type = 'merch_purchase' THEN 1 END)
            AS merch_purchase_count,
        COUNT(CASE WHEN event_type = 'ticket_scan' THEN 1 END)
            AS ticket_scan_count
    FROM match_events
    GROUP BY fan_id
""")

db.commit()

# ── Summary stats ────────────────────────────────────────────────────
n_events = cur.execute("SELECT COUNT(*) FROM match_events").fetchone()[0]
n_fans = cur.execute("SELECT COUNT(*) FROM fan_loyalty").fetchone()[0]
n_scans = cur.execute(
    "SELECT COUNT(*) FROM match_events WHERE event_type='ticket_scan'"
).fetchone()[0]
n_merch = cur.execute(
    "SELECT COUNT(*) FROM match_events WHERE event_type='merch_purchase'"
).fetchone()[0]

print(f"\n✅ SQLite database ready (in-memory)")
print(f"   match_events: {n_events:,} rows ({n_scans:,} scans + {n_merch:,} merch)")
print(f"   fan_loyalty:  {n_fans:,} unique fans")

# Quick peek
print(f"\n   Sample from fan_loyalty (top 3 by spend):")
for row in cur.execute(
    "SELECT * FROM fan_loyalty ORDER BY total_spend DESC LIMIT 3"
).fetchall():
    print(f"     {dict(row)}")

---
## 1 · Schema Discovery Tools

The production SQL agent discovers the database schema on demand using
read-only tools. Below we build **SQLite equivalents** of each discovery
tool so we can demonstrate the same workflow without Postgres.

| Tool | Production (Postgres) | This notebook (SQLite) |
|------|----------------------|----------------------|
| `list_tables()` | `information_schema.tables` | `sqlite_master` |
| `describe_table(t)` | `information_schema.columns` | `PRAGMA table_info(t)` |
| `search_columns(p)` | `column_name ILIKE %p%` | iterate PRAGMAs + Python filter |
| `sample_table(t, n)` | `SELECT * FROM t LIMIT n` | same |

In [ ]:
import re

_VALID_IDENT = re.compile(r"^[A-Za-z_][A-Za-z0-9_]*$")


def list_tables() -> list[dict[str, str]]:
    """List all tables in the SQLite database."""
    rows = db.execute(
        "SELECT name, type FROM sqlite_master "
        "WHERE type IN ('table', 'view') AND name NOT LIKE 'sqlite_%' "
        "ORDER BY name"
    ).fetchall()
    return [{"name": r["name"], "type": r["type"]} for r in rows]


def describe_table(table: str) -> dict:
    """Describe one table: column name, type, nullable."""
    if not _VALID_IDENT.match(table):
        return {"error": f"Invalid identifier: {table!r}"}
    known = {t["name"] for t in list_tables()}
    if table not in known:
        return {"error": f"Unknown table {table!r}. Call list_tables() to see options."}
    rows = db.execute(f"PRAGMA table_info({table})").fetchall()
    columns = []
    for r in rows:
        columns.append({
            "name": r["name"],
            "data_type": r["type"] or "ANY",
            "nullable": r["notnull"] == 0,
            "pk": bool(r["pk"]),
        })
    return {"name": table, "columns": columns}


def search_columns(pattern: str) -> list[dict[str, str]]:
    """Search column names across all tables (case-insensitive substring match)."""
    pat = pattern.lower()
    results = []
    for tbl in list_tables():
        for col in describe_table(tbl["name"])["columns"]:
            if pat in col["name"].lower():
                results.append({
                    "table": tbl["name"],
                    "column": col["name"],
                    "data_type": col["data_type"],
                })
    return results


def sample_table(table: str, limit: int = 5) -> list[dict]:
    """Return a small sample of rows from one table."""
    if not _VALID_IDENT.match(table):
        return [{"error": f"Invalid identifier: {table!r}"}]
    cap = max(1, min(limit, 10))
    rows = db.execute(f"SELECT * FROM {table} LIMIT {cap}").fetchall()
    return [dict(r) for r in rows]


print("✅ SQLite discovery tools defined: list_tables, describe_table, search_columns, sample_table")

In [ ]:
print("=" * 60)
print("TOOL: list_tables()")
print("=" * 60)
print("\nThe agent's first step — discover what tables exist.\n")

tables = list_tables()
print(f"Found {len(tables)} tables in SQLite:\n")
print(f"  {'Name':<20} {'Type'}")
print(f"  {'-' * 20} {'-' * 10}")
for t in tables:
    print(f"  {t['name']:<20} {t['type']}")

In [ ]:
print("=" * 60)
print("TOOL: describe_table('fan_loyalty')")
print("=" * 60)
print("\nThe agent inspects columns, types, and constraints before writing SQL.\n")

desc = describe_table("fan_loyalty")
print(f"  Table: {desc['name']}\n")
print(f"  {'Column':<25} {'Type':<10} {'Nullable':<10} {'PK'}")
print(f"  {'-' * 25} {'-' * 10} {'-' * 10} {'-' * 5}")
for col in desc["columns"]:
    print(f"  {col['name']:<25} {col['data_type']:<10} {str(col['nullable']):<10} {col['pk']}")

print("\n" + "=" * 60)
print("TOOL: describe_table('match_events')")
print("=" * 60 + "\n")

desc2 = describe_table("match_events")
print(f"  Table: {desc2['name']}\n")
print(f"  {'Column':<25} {'Type':<10} {'Nullable':<10} {'PK'}")
print(f"  {'-' * 25} {'-' * 10} {'-' * 10} {'-' * 5}")
for col in desc2["columns"]:
    print(f"  {col['name']:<25} {col['data_type']:<10} {str(col['nullable']):<10} {col['pk']}")

In [ ]:
print("=" * 60)
print("TOOL: search_columns('spend')")
print("=" * 60)
print("\nSearch column names across ALL tables (case-insensitive).\n")

results = search_columns("spend")
print(f"  Found {len(results)} columns matching 'spend':\n")
print(f"  {'Table':<20} {'Column':<25} {'Type'}")
print(f"  {'-' * 20} {'-' * 25} {'-' * 10}")
for r in results:
    print(f"  {r['table']:<20} {r['column']:<25} {r['data_type']}")

# Also search for 'fan' to show cross-table results
print("\n" + "=" * 60)
print("TOOL: search_columns('fan')")
print("=" * 60 + "\n")

results2 = search_columns("fan")
print(f"  Found {len(results2)} columns matching 'fan':\n")
for r in results2:
    print(f"  {r['table']:<20} {r['column']:<25} {r['data_type']}")

In [ ]:
print("=" * 60)
print("TOOL: sample_table('fan_loyalty', limit=5)")
print("=" * 60)
print("\nPeek at actual rows to verify data shape.\n")

sample = sample_table("fan_loyalty", limit=5)
if sample:
    cols = list(sample[0].keys())
    header = "  " + " | ".join(f"{c:<20}" for c in cols)
    print(header)
    print("  " + "-" * len(header))
    for row in sample:
        vals = " | ".join(f"{str(row.get(c, '')):<20}" for c in cols)
        print(f"  {vals}")

print("\n" + "=" * 60)
print("TOOL: sample_table('match_events', limit=3)")
print("=" * 60 + "\n")

sample2 = sample_table("match_events", limit=3)
if sample2:
    cols2 = list(sample2[0].keys())
    header2 = "  " + " | ".join(f"{c:<15}" for c in cols2)
    print(header2)
    print("  " + "-" * len(header2))
    for row in sample2:
        vals = " | ".join(f"{str(row.get(c, '')):<15}" for c in cols2)
        print(f"  {vals}")

---
## 2 · SQL Guardrails

Before any SQL reaches the database it passes through **three layers** of
sanitisation (all in `guardrails.py` — imported directly, they are backend-agnostic):

```
Raw LLM output
      │
      ▼
┌─────────────────────────────────────┐
│  1. _strip_fences()                 │  Remove ```sql ... ``` wrappers
├─────────────────────────────────────┤
│  2. _rewrite_layer_schema_qualifiers│  marts.X → dbt_dev.X
├─────────────────────────────────────┤
│  3. _validate_sql()                 │  Two-pass validation:
│     ├─ sqlglot AST check            │   parse → reject DDL/DML nodes
│     └─ Regex belt-and-braces        │   catch mutating keywords
└─────────────────────────────────────┘
      │
      ▼
  Clean, validated SQL → database
```

In [ ]:
from frontend_app.sql_agent.guardrails import (
    _rewrite_layer_schema_qualifiers,
    _strip_fences,
    _validate_sql,
)

# ── Layer 1: Strip code fences ─────────────────────────────────────
print("=" * 60)
print("GUARDRAIL 1: _strip_fences()")
print("=" * 60)
print("\nLLMs often wrap SQL in markdown code fences.\n")

raw_from_llm = """```sql
SELECT fan_id, total_spend
FROM marts.fan_loyalty
ORDER BY total_spend DESC
LIMIT 5;
```"""

print("  INPUT (raw LLM output):")
for line in raw_from_llm.strip().split("\n"):
    print(f"    {line}")

step1 = _strip_fences(raw_from_llm)
print(f"\n  OUTPUT (fences + trailing semicolons removed):")
for line in step1.strip().split("\n"):
    print(f"    {line}")

# ── Layer 2: Rewrite schema qualifiers ────────────────────────────
print("\n" + "=" * 60)
print("GUARDRAIL 2: _rewrite_layer_schema_qualifiers()")
print("=" * 60)
print("\nRewrites marts.X → dbt_dev.X (smaller LLMs confuse layer names with schemas).\n")

step2 = _rewrite_layer_schema_qualifiers(step1)
print(f"  INPUT:  FROM marts.fan_loyalty")
print(f"  OUTPUT: FROM dbt_dev.fan_loyalty")
print(f"\n  Full rewritten SQL:")
for line in step2.strip().split("\n"):
    print(f"    {line}")

# ── Layer 3: Validate SQL (AST + regex) ──────────────────────────
print("\n" + "=" * 60)
print("GUARDRAIL 3a: _validate_sql() — SAFE query passes")
print("=" * 60)

safe_sql = "SELECT fan_id, total_spend FROM fan_loyalty ORDER BY total_spend DESC"
print(f"\n  SQL: {safe_sql}")
print(f"\n  Running validation…")

try:
    _validate_sql(safe_sql)
    print("    ├─ sqlglot parse:     ✅ parsed OK")
    print("    ├─ AST node check:    ✅ clean")
    print("    └─ Regex check:       ✅ clean")
    print("\n  ✅ Query passed all guardrails — safe to execute")
except ValueError as exc:
    print(f"  ❌ {exc}")

# ── Layer 3b: Dangerous queries blocked ─────────────────────────
print("\n" + "=" * 60)
print("GUARDRAIL 3b: _validate_sql() — DANGEROUS queries blocked")
print("=" * 60)
print("\nEach is rejected by the AST check, the regex check, or both.\n")

dangerous = [
    ("DROP TABLE fan_loyalty", "DDL"),
    ("SELECT * FROM fans; DELETE FROM fans", "Multi-statement + DML"),
    ("INSERT INTO fans VALUES (1, 'hacker')", "DML"),
    ("UPDATE fans SET name='pwned' WHERE 1=1", "DML"),
    ("TRUNCATE TABLE fan_loyalty", "DDL"),
]

for sql, category in dangerous:
    print(f"  SQL: {sql}")
    print(f"  Category: {category}")
    try:
        _validate_sql(sql)
        print(f"  Result: ⚠️  PASSED (unexpected!)")
    except ValueError as exc:
        print(f"  Result: 🛡️  BLOCKED")
        print(f"  Reason: {exc}")
    print()

---
## 3 · Safe SQL Execution — `execute_select_sqlite`

This is the **only path** for running model-generated SQL. It chains all
three guardrail layers, wraps the query in an outer `LIMIT 100`, and
executes against SQLite.

The function returns structured JSON — identical contract to the production
`execute_select` tool:
- **Success**: `{"rows": [...], "row_count": N, "sql": "..."}`
- **Validation error**: `{"error": "...", "phase": "validation", "sql": "..."}`
- **Execution error**: `{"error": "...", "phase": "execution", "sql": "..."}`

In [ ]:
import time


def execute_select_sqlite(sql: str) -> dict:
    """Execute a validated read-only SELECT against SQLite with full guardrails.

    Applies the same three-layer pipeline as production:
    1. Strip markdown fences
    2. Rewrite layer schema qualifiers
    3. Validate SQL (sqlglot AST + regex)
    Then wraps in LIMIT 100 and executes.
    """
    if not isinstance(sql, str) or not sql.strip():
        return {"error": "sql must be a non-empty string", "phase": "validation"}

    # Guardrail layer 1 + 2: strip fences and rewrite schema prefixes.
    cleaned = _rewrite_layer_schema_qualifiers(_strip_fences(sql))

    # For SQLite: also strip any remaining dbt_dev. qualifiers since
    # our tables live in the default schema, not a named one.
    cleaned = re.sub(r"\bdbt_dev\.", "", cleaned)

    # Guardrail layer 3: sqlglot AST + regex validation.
    try:
        _validate_sql(cleaned)
    except ValueError as exc:
        return {"error": str(exc), "phase": "validation", "sql": cleaned}

    # Wrap in LIMIT 100 safety net and execute.
    wrapped = f"SELECT * FROM ({cleaned}) LIMIT 100"

    t0 = time.perf_counter()
    try:
        rows = [dict(r) for r in db.execute(wrapped).fetchall()]
    except Exception as exc:
        return {"error": str(exc), "phase": "execution", "sql": cleaned}
    elapsed_ms = (time.perf_counter() - t0) * 1000

    return {"rows": rows, "row_count": len(rows), "sql": cleaned, "elapsed_ms": round(elapsed_ms)}


# ── Demo: valid query ─────────────────────────────────────────────
print("=" * 60)
print("execute_select_sqlite() — valid query")
print("=" * 60)

sql = (
    "SELECT fan_id, total_spend, matches_attended "
    "FROM fan_loyalty ORDER BY total_spend DESC LIMIT 5"
)
print(f"\n  SQL: {sql}")
print(f"  Pipeline: strip_fences → rewrite_schema → validate_sql → execute")

result = execute_select_sqlite(sql)

if "error" in result:
    print(f"\n  ❌ {result['phase']}: {result['error']}")
else:
    print(f"\n  ✅ Query succeeded!")
    print(f"     Rows returned: {result['row_count']}")
    print(f"     Elapsed:       {result['elapsed_ms']} ms")
    print(f"\n     Top 5 fans by spend:")
    print(f"     {'fan_id':<15} {'total_spend':>12} {'matches_attended':>18}")
    print(f"     {'-' * 47}")
    for row in result["rows"][:5]:
        print(f"     {row['fan_id']:<15} {row['total_spend']:>12.2f} {row['matches_attended']:>18}")

# ── Demo: blocked query ───────────────────────────────────────────
print("\n" + "=" * 60)
print("execute_select_sqlite() — blocked query")
print("=" * 60)

bad_sql = "DROP TABLE fan_loyalty"
print(f"\n  SQL: {bad_sql}")
result_bad = execute_select_sqlite(bad_sql)
print(f"  Phase:  {result_bad['phase']}")
print(f"  Error:  {result_bad['error']}")
print(f"\n  ✅ Dangerous query was blocked before reaching the database!")

# ── Demo: LLM schema mistake auto-corrected ──────────────────────
print("\n" + "=" * 60)
print("execute_select_sqlite() — LLM schema mistake auto-corrected")
print("=" * 60)

fenced_sql = """```sql
SELECT fan_id, total_spend FROM marts.fan_loyalty ORDER BY total_spend DESC LIMIT 3
```"""
print(f"\n  SQL (raw from LLM): marts.fan_loyalty wrapped in fences")
result_fixed = execute_select_sqlite(fenced_sql)

if "error" in result_fixed:
    print(f"  ❌ {result_fixed['phase']}: {result_fixed['error']}")
else:
    print(f"  ✅ Auto-corrected! Executed: {result_fixed['sql']}")
    print(f"     Rows: {result_fixed['row_count']}")

---
## 4 · Full Tool Walkthrough

This cell simulates the agent's natural discovery flow — the same sequence of
tool calls the ReAct loop would make when answering a question like:

> *"Who are the top 3 fans by total spend, and how many matches did each attend?"*

```
list_tables → describe_table → search_columns → sample_table → execute_select
```

In [ ]:
question = "Who are the top 3 fans by total spend, and how many matches did each attend?"

print("=" * 60)
print("AGENT TOOL-CALL WALKTHROUGH")
print("=" * 60)
print(f"\n  Question: {question}")
print("\n  Simulating the agent's ReAct reasoning steps…\n")

# Step 1: Discover tables
print("─" * 60)
print("Step 1 → list_tables()")
tables = list_tables()
table_names = [t["name"] for t in tables]
print(f"  Agent sees: {table_names}")
print(f"  Reasoning: fan_loyalty looks like the right table for fan spend data\n")

# Step 2: Inspect fan_loyalty
print("─" * 60)
print("Step 2 → describe_table('fan_loyalty')")
desc = describe_table("fan_loyalty")
col_names = [c["name"] for c in desc["columns"]]
print(f"  Agent sees columns: {col_names}")
print(f"  Reasoning: total_spend and matches_attended are exactly what I need\n")

# Step 3: Verify with search_columns
print("─" * 60)
print("Step 3 → search_columns('spend')")
spend_cols = search_columns("spend")
for r in spend_cols:
    print(f"  {r['table']}.{r['column']} ({r['data_type']})")
print(f"  Reasoning: total_spend and merch_spend are both in fan_loyalty — confirmed\n")

# Step 4: Peek at sample rows
print("─" * 60)
print("Step 4 → sample_table('fan_loyalty', limit=3)")
sample = sample_table("fan_loyalty", limit=3)
for row in sample:
    print(f"  {row}")
print(f"  Reasoning: data looks reasonable, ready to write the final query\n")

# Step 5: Execute the query
print("─" * 60)
print("Step 5 → execute_select_sqlite()")
final_sql = """
SELECT fan_id, total_spend, matches_attended
FROM fan_loyalty
ORDER BY total_spend DESC
LIMIT 3
"""
print(f"  SQL:\n    {final_sql.strip()}")
result = execute_select_sqlite(final_sql)

if "error" in result:
    print(f"\n  ❌ {result['phase']}: {result['error']}")
else:
    print(f"\n  ✅ Query succeeded — {result['row_count']} rows\n")
    print(f"  FINAL ANSWER:")
    print(f"  ┌{"─" * 50}┐")
    print(f"  │ {'Fan ID':<15} {'Total Spend (€)':>17} {'Matches':>14} │")
    print(f"  ├{"─" * 50}┤")
    for row in result["rows"]:
        print(f"  │ {row['fan_id']:<15} {row['total_spend']:>17.2f} {row['matches_attended']:>14} │")
    print(f"  └{"─" * 50}┘")

---
## 5 · ReAct Agent Loop *(optional — requires OpenRouter API key)*

The cell below wires up the SQLite discovery tools as real LangChain tools
and runs a full ReAct agent loop. The agent will:

1. Discover the schema (`list_tables`, `describe_table`)
2. Write and validate SQL (`execute_select_sqlite`)
3. Produce a Markdown answer from the returned rows

> **Requires** `OPENROUTER_API_KEY` set in your `.env` file.
> If the key is missing, the cell skips gracefully.

In [ ]:
import os

from dotenv import load_dotenv

load_dotenv(dotenv_path=REPO_ROOT / ".env")

if not os.getenv("OPENROUTER_API_KEY"):
    print("⚠️  OPENROUTER_API_KEY not set — skipping live agent demo.")
    print("   Set it in your .env file and rerun this cell.")
else:
    from langchain.agents import create_agent
    from langchain_core.messages import AIMessage, HumanMessage, ToolMessage
    from langchain_core.tools import tool

    from frontend_app.sql_agent.llm_runtime_config import (
        init_llm_config,
        resolve_agent_model,
    )
    from frontend_app.sql_agent.observability import AgentObservabilityHandler
    from frontend_app.sql_agent.providers import build_chat_model

    init_llm_config()

    # Wrap local functions as LangChain tools for the agent.
    @tool
    def agent_list_tables() -> str:
        """List all tables available in the database."""
        return json.dumps(list_tables())

    @tool
    def agent_describe_table(table: str) -> str:
        """Describe columns of one table (name, type, nullable)."""
        return json.dumps(describe_table(table))

    @tool
    def agent_search_columns(pattern: str) -> str:
        """Search column names across all tables (case-insensitive)."""
        return json.dumps(search_columns(pattern))

    @tool
    def agent_sample_table(table: str, limit: int = 5) -> str:
        """Return a small sample of rows from one table."""
        return json.dumps(sample_table(table, limit))

    @tool
    def agent_execute_select(sql: str) -> str:
        """Execute a validated read-only SELECT query and return JSON rows.

        The SQL is sanitised and validated before execution. Returns
        {"rows": [...], "row_count": N} on success, or
        {"error": "...", "phase": "validation"|"execution"} on failure.
        """
        return json.dumps(execute_select_sqlite(sql))

    all_tools = [
        agent_list_tables,
        agent_describe_table,
        agent_search_columns,
        agent_sample_table,
        agent_execute_select,
    ]

    SYSTEM_PROMPT = """\
You are a careful SQL data analyst for a football club's analytics database.
You answer the user's question by:

1. Discovering what data is available via the supplied tools.
2. Writing a single read-only SELECT statement.
3. Executing it via the `agent_execute_select` tool.
4. Producing a short, clear Markdown answer using the returned rows.

TOOLS:
- `agent_list_tables()` — list every table.
- `agent_describe_table(table)` — columns for one table.
- `agent_search_columns(pattern)` — find columns by name.
- `agent_sample_table(table, limit)` — peek at rows.
- `agent_execute_select(sql)` — the ONLY way to run SQL.

RULES:
- Only SELECT statements. No DDL, no DML, no semicolons.
- Use unqualified table names (no schema prefix).
- On validation/execution errors, fix the SQL and retry.
"""

    agent_model_id = resolve_agent_model()
    llm = build_chat_model(agent_model_id)
    agent_question = (
        "Who are the top 3 fans by total spend, "
        "and how many matches did each attend?"
    )

    print("=" * 60)
    print("REACT AGENT LOOP")
    print("=" * 60)
    print(f"\n  Question: {agent_question}")
    print(f"  Model:    {agent_model_id}")
    print(f"  Tools:    {', '.join(t.name for t in all_tools)}")

    agent = create_agent(
        model=llm, tools=all_tools, system_prompt=SYSTEM_PROMPT
    )

    print(f"\n  Running agent…\n")
    handler = AgentObservabilityHandler()
    state = agent.invoke(
        {"messages": [HumanMessage(content=agent_question)]},
        config={"recursion_limit": 25, "callbacks": [handler]},
    )

    # Display tool call trace and final answer.
    messages = state.get("messages", [])
    print("─" * 60)
    print("TOOL CALL TRACE:")
    print("─" * 60)
    for i, msg in enumerate(messages):
        if isinstance(msg, AIMessage) and getattr(msg, "tool_calls", None):
            for tc in msg.tool_calls:
                args_preview = json.dumps(tc.get("args", {}))[:80]
                print(f"  [{i}] 🤖 → {tc['name']}({args_preview})")
        elif isinstance(msg, ToolMessage):
            content_preview = (
                (msg.content[:100] + "…")
                if len(msg.content) > 100
                else msg.content
            )
            print(f"  [{i}] 🔧 {msg.name} → {content_preview}")

    # Extract and display the final answer.
    final_answer = ""
    for msg in reversed(messages):
        if isinstance(msg, AIMessage) and not getattr(
            msg, "tool_calls", None
        ):
            final_answer = (
                msg.content
                if isinstance(msg.content, str)
                else str(msg.content)
            )
            break

    print("\n" + "=" * 60)
    print("FINAL ANSWER")
    print("=" * 60)
    print(f"\n{final_answer}")

---
## Summary

This notebook demonstrated the complete SQL agent pipeline in a self-contained
environment:

| Component | What we showed |
|-----------|---------------|
| **Data generation** | Full-season synthetic fan events from `v2_calendar` generator |
| **Schema discovery** | `list_tables`, `describe_table`, `search_columns`, `sample_table` — all working against SQLite |
| **SQL guardrails** | Three-layer validation: fence stripping → schema rewriting → sqlglot AST + regex |
| **Safe execution** | `execute_select_sqlite` with LIMIT 100 safety net |
| **Agent loop** | *(optional)* Full ReAct agent answering a natural language question |

### Key takeaways

1. **Guardrails are backend-agnostic** — the same `_validate_sql` that protects
   Postgres in production works identically against SQLite
2. **Discovery tools adapt easily** — the pattern of querying metadata and
   returning structured JSON works for any SQL database
3. **The ReAct loop is model-agnostic** — any tool-calling LLM can drive the
   same agent pipeline

**Next steps:** See `notebooks/sql-agent.ipynb` for the full production pipeline
with PostgreSQL, the semantic layer, and the repair pass.